Week 2

In [1]:
import pandas as pd
import os

In [2]:

input_folder = "/Users/yangzewen/Desktop/IDX Summer 2026/Week 1"
output_folder = "/Users/yangzewen/Desktop/IDX Summer 2026/Week 2"
os.makedirs(output_folder, exist_ok=True)

sold_path = os.path.join(input_folder, "Final_Sold.csv")
listed_path = os.path.join(input_folder, "Final_Listed.csv")


1. Load datasets

In [3]:
sold = pd.read_csv(sold_path)
listed = pd.read_csv(listed_path)

print("Datasets loaded successfully")
print("Sold shape:", sold.shape)
print("Listed shape:", listed.shape)

# Keep original copies for comparison
sold_original = sold.copy()
listed_original = listed.copy()

/var/folders/c0/7228ggf57vddb8zg5691thww0000gn/T/ipykernel_5805/3781303924.py:1: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv(sold_path)
/var/folders/c0/7228ggf57vddb8zg5691thww0000gn/T/ipykernel_5805/3781303924.py:2: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listed = pd.read_csv(listed_path)


Datasets loaded successfully
Sold shape: (430566, 84)
Listed shape: (590771, 84)


2. Check PropertyType

In [ ]:
for name, df in {"Sold": sold, "Listed": listed}.items():
    print(f"\nPropertyType distribution ({name}):")
    print(df["PropertyType"].value_counts(dropna=False))


PropertyType distribution (Sold):
PropertyType
Residential    430566
Name: count, dtype: int64

PropertyType distribution (Listed):
PropertyType
Residential    590771
Name: count, dtype: int64


3. Clean and filter Residential properties

In [5]:
for df in [sold, listed]:
    df["PropertyType"] = df["PropertyType"].astype(str).str.strip()

assert (sold["PropertyType"] == "Residential").all(), "sold.csv contains non-Residential rows."
assert (listed["PropertyType"] == "Residential").all(), "listed.csv contains non-Residential rows."

print("\nAfter filtering Residential:")
print("Sold shape:", sold.shape)
print("Listed shape:", listed.shape)


After filtering Residential:
Sold shape: (430566, 84)
Listed shape: (590771, 84)


In [ ]:
4. Missing value report

In [7]:
def missing_report(df, name):
    report = pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_percent": df.isnull().mean() * 100
    })

    report["flag_90pct_missing"] = report["missing_percent"] > 90
    report = report.sort_values("missing_percent", ascending=False)

    print(f"\nMissing Value Report: {name}")
    print(report.head(20))

    return report

missing_sold = missing_report(sold, "Sold")
missing_listed = missing_report(listed, "Listed")

missing_sold.to_csv(os.path.join(output_folder, "Sold_missing_report.csv"))
missing_listed.to_csv(os.path.join(output_folder, "Listed_missing_report.csv"))


Missing Value Report: Sold
                              missing_count  missing_percent  \
CoveredSpaces                        430566       100.000000   
MiddleOrJuniorSchoolDistrict         430566       100.000000   
AboveGradeFinishedArea               430566       100.000000   
FireplacesTotal                      430566       100.000000   
TaxYear                              430566       100.000000   
ElementarySchoolDistrict             430566       100.000000   
BusinessType                         430566       100.000000   
TaxAnnualAmount                      430566       100.000000   
WaterfrontYN                         430295        99.937060   
BelowGradeFinishedArea               428039        99.413098   
BasementYN                           422126        98.039789   
LotSizeDimensions                    409654        95.143137   
BuilderName                          409488        95.104583   
BuildingAreaTotal                    400371        92.987138   
CoBuyerAgent

5. Numeric summary for key columns

In [8]:
key_numeric_columns = ["ClosePrice", "LivingArea", "DaysOnMarket"]

for col in key_numeric_columns:
    if col in sold.columns:
        print(f"\nSummary Statistics for Sold - {col}")
        print(sold[col].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))


Summary Statistics for Sold - ClosePrice
count    4.305640e+05
mean     1.190194e+06
std      6.026882e+06
min      0.000000e+00
1%       2.020000e+05
5%       3.400000e+05
25%      5.750000e+05
50%      8.250000e+05
75%      1.300000e+06
95%      2.850000e+06
99%      5.575000e+06
max      9.895000e+08
Name: ClosePrice, dtype: float64

Summary Statistics for Sold - LivingArea
count    4.303210e+05
mean     1.904044e+03
std      2.596437e+04
min      0.000000e+00
1%       6.050000e+02
5%       8.400000e+02
25%      1.248000e+03
50%      1.644000e+03
75%      2.221000e+03
95%      3.562000e+03
99%      5.283800e+03
max      1.702132e+07
Name: LivingArea, dtype: float64

Summary Statistics for Sold - DaysOnMarket
count    430566.000000
mean         37.339662
std          53.674388
min        -288.000000
1%            0.000000
5%            1.000000
25%           8.000000
50%          18.000000
75%          48.000000
95%         132.000000
99%         232.000000
max       12430.000000
Na

6. Final summary

In [9]:
summary = pd.DataFrame({
    "Dataset": ["Listed", "Sold"],
    "Rows_before_filter": [len(listed_original), len(sold_original)],
    "Rows_after_filter": [len(listed), len(sold)],
    "Rows_removed": [len(listed_original) - len(listed), len(sold_original) - len(sold)],
    "Columns": [listed.shape[1], sold.shape[1]]
})

print("\nData Cleaning Summary:")
print(summary)

summary.to_csv(os.path.join(output_folder, "Cleaning_summary.csv"), index=False)


Data Cleaning Summary:
  Dataset  Rows_before_filter  Rows_after_filter  Rows_removed  Columns
0  Listed              590771             590771             0       84
1    Sold              430566             430566             0       84


7. Save cleaned datasets

In [10]:
sold.to_csv(os.path.join(output_folder, "Sold_filtered.csv"), index=False)
listed.to_csv(os.path.join(output_folder, "Listed_filtered.csv"), index=False)

print("\nFiltered datasets and reports saved successfully.")


Filtered datasets and reports saved successfully.


Mortgage Rate Enrichment- Week 3

1. Fetch mortgage rate data from FRED

In [11]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url)

mortgage.columns = ['date', 'rate_30yr_fixed']

mortgage['date'] = pd.to_datetime(

    mortgage['date'],

    errors='coerce'

)

mortgage['rate_30yr_fixed'] = pd.to_numeric(

    mortgage['rate_30yr_fixed'],

    errors='coerce'

)

mortgage = mortgage.dropna(

    subset=['date', 'rate_30yr_fixed']

)

2. Convert weekly data to monthly averages

In [12]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')

mortgage_monthly = (

    mortgage

    .groupby('year_month', as_index=False)['rate_30yr_fixed']

    .mean()

)

In [ ]:
3.  Create matching keys for merge

In [13]:
sold['CloseDate'] = pd.to_datetime(

    sold['CloseDate'],

    errors='coerce'

)

listed['ListingContractDate'] = pd.to_datetime(

    listed['ListingContractDate'],

    errors='coerce'

)

sold['year_month'] = sold['CloseDate'].dt.to_period('M')

listed['year_month'] = (

    listed['ListingContractDate'].dt.to_period('M')

)

In [ ]:
4. Merge mortgage data

In [14]:
sold_with_rates = sold.merge(

    mortgage_monthly,

    on='year_month',

    how='left',

    validate='many_to_one'

)

listed_with_rates = listed.merge(

    mortgage_monthly,

    on='year_month',

    how='left',

    validate='many_to_one'

)

In [19]:
# Validate merge results
missing_sold = sold_with_rates['rate_30yr_fixed'].isnull().sum()
missing_listed = listed_with_rates['rate_30yr_fixed'].isnull().sum()

print("\nMissing mortgage rates in SOLD:", missing_sold)
print("Missing mortgage rates in LISTED:", missing_listed)


Missing mortgage rates in SOLD: 0
Missing mortgage rates in LISTED: 0


Save final datasets

In [23]:
assert len(sold_with_rates) == len(sold)

assert len(listed_with_rates) == len(listed)

output_dir = "/Users/yangzewen/Desktop/IDX Summer 2026/Week 2"


sold_with_rates.to_csv(
    f"{output_dir}/Sold_Processed.csv",
    index=False
)

listed_with_rates.to_csv(
    f"{output_dir}/Listed_Processed.csv",
    index=False
)

print("\nWeek 2–3 processing completed successfully")


Week 2–3 processing completed successfully
